# 05 — Quantitative figures: scatter, SRCC per type, monotonicity, SSIM variants (B-09, B-06, B-22)

Inputs: `results/scores.csv` (B-07), `results/ssim_variants.csv` (`scripts/ssim_variants.py`, B-22).
Outputs: `results/figures/quant/*.png`, `results/tables/{monotonicity_by_type,ssim_variants}.csv`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from humper.data import REPO_ROOT
from humper.evaluate import correlations, monotonicity, srcc, table
from humper.viz import LABELS, scatter_figure, type_heatmap

FIG = REPO_ROOT / "results" / "figures" / "quant"; FIG.mkdir(parents=True, exist_ok=True)
TAB = REPO_ROOT / "results" / "tables"
s = pd.read_csv(REPO_ROOT / "results" / "scores.csv")

## Overall SRCC / PLCC (all 10,125 pairs, direction-aligned)

In [ ]:
overall = table(s).set_index("metric")[["srcc", "krcc", "plcc_raw", "plcc"]].round(3)
overall

## Scatter: raw metric vs MOS (native direction; black = logistic used for PLCC)

In [ ]:
fig = scatter_figure(s)
fig.savefig(FIG / "scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## SRCC within each distortion type

In [ ]:
fig = type_heatmap(pd.read_csv(TAB / "srcc_by_type.csv", index_col=0))
fig.savefig(FIG / "srcc_by_type.png", dpi=150, bbox_inches="tight")
plt.show()

## B-06 Monotonicity vs level

`share` = fraction of the 81 (reference, type) series whose aligned score never improves as the level rises; `rho` = mean Spearman(level, score), -1 = perfectly monotone.
KADID levels are **not** a severity order for `mean_shift` and `contrast_change`: those levels go in both directions around the reference (e.g. mean shift level 3 is the identity). Any metric, and MOS too, fails there by construction.

In [ ]:
mono = monotonicity(s)
mono.round(4).to_csv(TAB / "monotonicity_by_type.csv", index=False)
share = mono.pivot(index="dist_name", columns="metric", values="share")[["mos", "psnr", "ssim", "ms_ssim", "lpips_vgg", "dists"]]
display(share.round(2))
display(mono.groupby("metric")[["rho", "share"]].mean().round(3))

## B-22 SSIM variants (D-20)

Same formula, one change at a time: unclamped contrast-structure term (textbook / skimage), Wang's 2x downsampling, both.

In [ ]:
v = s[["id", "mos", "ssim", "dist_group"]].merge(pd.read_csv(REPO_ROOT / "results" / "ssim_variants.csv"), on="id")
names = ["ssim", "ssim_noclamp", "ssim_ds", "ssim_ds_noclamp"]
# every variant is a similarity (higher = better), so no direction flip is needed
var = pd.DataFrame({n: correlations(v[n], v["mos"]) for n in names}).T[["srcc", "plcc"]].astype(float)
var["max |Δ| vs main"] = [0.0] + [float((v[n] - v["ssim"]).abs().max()) for n in names[1:]]
by_group = pd.DataFrame({n: v.groupby("dist_group").apply(lambda g: srcc(g[n], g["mos"]), include_groups=False) for n in names})
var.round(4).to_csv(TAB / "ssim_variants.csv")
display(var.round(4)); display(by_group.round(3))